# Electricity price thresholds for the EMS

Goal: convert the spot price data into low, medium, and high price states.

The raw file stores prices as DKK/MWh with comma decimals. The app uses DKK/kWh, so the formula is:

price_DKK_per_kWh = price_DKK_per_MWh / 1000

I use the 25 percent and 75 percent quantiles as the two state thresholds.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This makes the notebook work both from the repo root and from its own folder.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not find the project root folder containing data/")
    PROJECT_ROOT = PROJECT_ROOT.parent

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)


## 1. Load and inspect the raw price data


In [ ]:
price_file = PROJECT_ROOT / "app/python/data/price_data/Elspotprices.csv"

prices_raw = pd.read_csv(price_file, sep=";", decimal=",")

print("File:", price_file)
print("Rows:", len(prices_raw))
display(prices_raw.head())
display(prices_raw.describe(include="all"))


## 2. Convert the price unit


In [ ]:
prices = prices_raw.copy()
prices["SpotPriceDKK_per_kWh"] = prices["SpotPriceDKK"].astype(float) / 1000

display(prices[["HourDK", "PriceArea", "SpotPriceDKK", "SpotPriceDKK_per_kWh"]].head())

print(f"Minimum price: {prices['SpotPriceDKK_per_kWh'].min():.4f} DKK/kWh")
print(f"Maximum price: {prices['SpotPriceDKK_per_kWh'].max():.4f} DKK/kWh")


## 3. Calculate low, medium, and high price split points

These are relative thresholds for the selected price dataset. A negative low price is possible and is
physically meaningful in electricity markets.


In [ ]:
price_DKK_per_kWh = prices["SpotPriceDKK_per_kWh"].dropna()

LOW_PRICE_THRESHOLD_DKK_PER_KWH = price_DKK_per_kWh.quantile(0.25)
HIGH_PRICE_THRESHOLD_DKK_PER_KWH = price_DKK_per_kWh.quantile(0.75)

print(f"Low/medium threshold:  {LOW_PRICE_THRESHOLD_DKK_PER_KWH:.4f} DKK/kWh")
print(f"Medium/high threshold: {HIGH_PRICE_THRESHOLD_DKK_PER_KWH:.4f} DKK/kWh")


## 4. Check the price distribution


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(price_DKK_per_kWh, bins=30, edgecolor="black", alpha=0.75)
plt.axvline(LOW_PRICE_THRESHOLD_DKK_PER_KWH, color="tab:green", linestyle="--", label="low/medium")
plt.axvline(HIGH_PRICE_THRESHOLD_DKK_PER_KWH, color="tab:red", linestyle="--", label="medium/high")
plt.title("Electricity price distribution")
plt.xlabel("Electricity price [DKK/kWh]")
plt.ylabel("Number of hours")
plt.legend()
plt.grid(True)
plt.show()


## 5. Values to use in the app


In [ ]:
price_parameters = pd.DataFrame(
    {
        "parameter": [
            "LOW_PRICE_THRESHOLD_DKK_PER_KWH",
            "HIGH_PRICE_THRESHOLD_DKK_PER_KWH",
        ],
        "value": [
            LOW_PRICE_THRESHOLD_DKK_PER_KWH,
            HIGH_PRICE_THRESHOLD_DKK_PER_KWH,
        ],
        "unit": ["DKK/kWh", "DKK/kWh"],
        "meaning": [
            "Below this is low price",
            "Above this is high price",
        ],
    }
)

display(price_parameters)
